# 第4章 多层感知机（MLP）

本章从线性模型出发，引入隐藏层与激活函数构建多层感知机，并系统讨论训练深层网络所需的核心技术：

| 节 | 内容 |
|---|---|
| 4.1 | 多层感知机原理：隐藏层、激活函数 |
| 4.2 | 简洁实现（PyTorch） |
| 4.3 | 模型选择、欠拟合与过拟合 |
| 4.4 | 权重衰减（L2 正则化） |
| 4.5 | Dropout |
| 4.6 | 前向传播与反向传播 |
| 4.7 | 数值稳定性与参数初始化 |
| 4.8 | 环境与分布偏移 |

> 参考：[动手学深度学习 v2 · 第4章](https://zh-v2.d2l.ai/chapter_multilayer-perceptrons/index.html)

In [ ]:
import torch
import torch.nn as nn
from torch.utils import data
import torchvision
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt

print(f'PyTorch 版本: {torch.__version__}')

---
## 4.1 多层感知机

### 4.1.1 线性模型的局限

单层线性模型（Softmax 回归）对输入做仿射变换：$\mathbf{o} = \mathbf{W}\mathbf{x} + \mathbf{b}$。

**关键问题**：仿射函数的仿射函数仍是仿射函数——无论堆叠多少线性层，等价于一个线性层，无法拟合非线性关系。

**解决方案**：在层间插入**非线性激活函数** $\sigma$，打破线性性。

### 4.1.2 MLP 数学表达

以单隐藏层 MLP 为例（输入维度 $d$，隐藏层 $h$ 个单元，输出 $q$ 类）：

$$\mathbf{H} = \sigma(\mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)})  \qquad \mathbf{H}\in\mathbb{R}^{n\times h}$$

$$\mathbf{O} = \mathbf{H}\mathbf{W}^{(2)} + \mathbf{b}^{(2)}  \qquad \mathbf{O}\in\mathbb{R}^{n\times q}$$

激活函数 $\sigma$ **逐元素**作用于隐藏层，赋予网络非线性表达能力。

### 4.1.3 激活函数

激活函数决定了网络的非线性特性，下面逐一介绍并可视化三种常用激活函数。

In [ ]:
x = torch.linspace(-8, 8, 400, requires_grad=True)

def plot_activation(ax_f, ax_d, y, title):
    """绘制激活函数及其导数"""
    y_sum = y.sum()
    y_sum.backward()
    grad = x.grad.detach().numpy()
    x.grad.zero_()

    xn = x.detach().numpy()
    yn = y.detach().numpy()
    ax_f.plot(xn, yn, color='steelblue')
    ax_f.set_title(title, fontsize=12)
    ax_f.axhline(0, color='k', lw=0.6, ls='--')
    ax_f.axvline(0, color='k', lw=0.6, ls='--')
    ax_f.set_ylabel('$f(x)$')
    ax_f.grid(True, alpha=0.3)

    ax_d.plot(xn, grad, color='tomato')
    ax_d.axhline(0, color='k', lw=0.6, ls='--')
    ax_d.axvline(0, color='k', lw=0.6, ls='--')
    ax_d.set_ylabel("$f'(x)$")
    ax_d.set_xlabel('$x$')
    ax_d.grid(True, alpha=0.3)

fig, axes = plt.subplots(2, 3, figsize=(13, 6))

# ReLU
plot_activation(axes[0,0], axes[1,0], torch.relu(x), 'ReLU')

# Sigmoid
plot_activation(axes[0,1], axes[1,1], torch.sigmoid(x), 'Sigmoid')

# Tanh
plot_activation(axes[0,2], axes[1,2], torch.tanh(x), 'Tanh')

fig.suptitle('激活函数（上：函数值，下：导数）', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**三种激活函数对比：**

| 激活函数 | 公式 | 值域 | 导数最大值 | 特点 |
|---------|------|------|-----------|------|
| **ReLU** | $\max(0, x)$ | $[0, +\infty)$ | 1 | 计算简单，缓解梯度消失；负区间梯度为 0（神经元死亡问题）|
| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | $(0, 1)$ | 0.25 | 输出可解释为概率；两端导数趋 0（梯度消失）|
| **Tanh** | $\frac{e^x - e^{-x}}{e^x + e^{-x}}$ | $(-1, 1)$ | 1 | 零中心化，比 sigmoid 收敛快；仍有梯度消失问题 |

---
## 4.2 MLP 的简洁实现（PyTorch）

使用 `nn.Sequential` 搭建两层 MLP，在 Fashion-MNIST 上进行图像分类。

### 4.2.1 加载数据

In [ ]:
def load_data_fashion_mnist(batch_size):
    trans = transforms.ToTensor()
    train_set = torchvision.datasets.FashionMNIST(
        root='../data', train=True,  transform=trans, download=True)
    test_set  = torchvision.datasets.FashionMNIST(
        root='../data', train=False, transform=trans, download=True)
    train_iter = data.DataLoader(train_set, batch_size, shuffle=True,  num_workers=0)
    test_iter  = data.DataLoader(test_set,  batch_size, shuffle=False, num_workers=0)
    return train_iter, test_iter

batch_size = 256
train_iter, test_iter = load_data_fashion_mnist(batch_size)

### 4.2.2 定义 MLP

In [ ]:
net = nn.Sequential(
    nn.Flatten(),          # (B, 1, 28, 28) -> (B, 784)
    nn.Linear(784, 256),   # 第一隐藏层
    nn.ReLU(),
    nn.Linear(256, 10)     # 输出层（10 类）
)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, std=0.01)
        nn.init.zeros_(m.bias)

net.apply(init_weights)
print(net)
print(f'\n可训练参数量: {sum(p.numel() for p in net.parameters()):,}')

### 4.2.3 训练与评估

In [ ]:
def accuracy(y_hat, y):
    return (y_hat.argmax(dim=1) == y).float().sum().item()

def evaluate_accuracy(net, data_iter):
    net.eval()
    correct, total = 0.0, 0
    with torch.no_grad():
        for X, y in data_iter:
            correct += accuracy(net(X), y)
            total += y.numel()
    return correct / total

def train(net, train_iter, test_iter, num_epochs, lr):
    loss_fn  = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(num_epochs):
        net.train()
        total_loss, correct, total = 0.0, 0.0, 0
        for X, y in train_iter:
            optimizer.zero_grad()
            y_hat = net(X)
            l = loss_fn(y_hat, y)
            l.backward()
            optimizer.step()
            total_loss += l.item() * y.numel()
            correct    += accuracy(y_hat, y)
            total      += y.numel()

        tr_loss = total_loss / total
        tr_acc  = correct / total
        te_acc  = evaluate_accuracy(net, test_iter)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['test_acc'].append(te_acc)
        print(f'epoch {epoch+1:2d}  loss {tr_loss:.4f}  '
              f'train acc {tr_acc:.3f}  test acc {te_acc:.3f}')
    return history

history_mlp = train(net, train_iter, test_iter, num_epochs=10, lr=0.1)

In [ ]:
epochs = range(1, 11)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(epochs, history_mlp['train_loss'], marker='o')
axes[0].set_title('训练损失')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history_mlp['train_acc'], marker='o', label='训练集')
axes[1].plot(epochs, history_mlp['test_acc'],  marker='s', label='测试集')
axes[1].set_title('分类准确率')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('MLP（单隐藏层 256 units，ReLU）on Fashion-MNIST', y=1.01)
plt.tight_layout(); plt.show()

---
## 4.3 模型选择、欠拟合与过拟合

### 4.3.1 核心概念

- **训练误差**：模型在训练集上的损失，可通过优化持续降低
- **泛化误差**：模型在未见数据上的期望损失，是真正关心的目标
- **欠拟合（underfitting）**：训练误差和测试误差都高 → 模型过于简单
- **过拟合（overfitting）**：训练误差 $\ll$ 测试误差 → 模型记住了训练集噪声

### 4.3.2 影响泛化的因素

| 因素 | 增大过拟合风险 |
|------|---------------|
| 参数数量 | 参数越多，风险越大 |
| 参数取值范围 | 权重值越大，风险越大 |
| 训练样本量 | 样本越少，风险越大 |

### 4.3.3 多项式回归实验

In [ ]:
# 生成合成数据：y = 5 + 1.2x - 3.4x²/2! + 5.6x³/3! + ε
max_degree = 20
n_train, n_test = 100, 100
true_w = np.zeros(max_degree)
true_w[0:4] = np.array([5, 1.2, -3.4, 5.6])

features = np.random.normal(0, 1, (n_train + n_test, 1))
# 构造多项式特征矩阵 [x^0/0!, x^1/1!, ..., x^19/19!]
poly_features = np.power(features, np.arange(max_degree).reshape(1, -1))
poly_features /= np.array([np.math.factorial(i) for i in range(max_degree)])
labels = poly_features @ true_w + np.random.normal(0, 0.1, (n_train + n_test,))

# 转换为张量
true_w_t   = torch.tensor(true_w[0:4], dtype=torch.float32)
features   = torch.tensor(features,      dtype=torch.float32)
poly_ft    = torch.tensor(poly_features, dtype=torch.float32)
labels     = torch.tensor(labels,        dtype=torch.float32)

train_poly, test_poly   = poly_ft[:n_train], poly_ft[n_train:]
train_labels, test_labels = labels[:n_train], labels[n_train:]
print('多项式特征矩阵形状:', poly_ft.shape)

In [ ]:
def evaluate_loss(net, data_iter, loss_fn):
    """在数据集上计算平均 MSE 损失"""
    total_loss, total = 0.0, 0
    net.eval()
    with torch.no_grad():
        for X, y in data_iter:
            l = loss_fn(net(X).squeeze(), y)
            total_loss += l.sum().item()
            total += y.numel()
    return total_loss / total

def train_poly(train_features, test_features, train_labels, test_labels,
               num_epochs=400, lr=0.01, batch_size=10):
    """训练线性模型拟合多项式特征"""
    loss_fn = nn.MSELoss(reduction='none')
    input_shape = train_features.shape[-1]
    net = nn.Linear(input_shape, 1, bias=False)
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)

    dataset = data.TensorDataset(train_features, train_labels)
    train_iter_ = data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    train_losses, test_losses = [], []
    net.train()
    for epoch in range(num_epochs):
        for X, y in train_iter_:
            optimizer.zero_grad()
            l = loss_fn(net(X).squeeze(), y)
            l.mean().backward()
            optimizer.step()

        if (epoch + 1) % 20 == 0:
            tr_l  = evaluate_loss(net,
                                  data.DataLoader(data.TensorDataset(train_features, train_labels), 100),
                                  nn.MSELoss(reduction='none'))
            te_l  = evaluate_loss(net,
                                  data.DataLoader(data.TensorDataset(test_features,  test_labels),  100),
                                  nn.MSELoss(reduction='none'))
            train_losses.append(tr_l)
            test_losses.append(te_l)

    print(f'权重后4项: {net.weight.data[:, :4]}')
    return train_losses, test_losses

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

configs = [
    ('正常拟合（3阶）',   train_poly[:, :4], test_poly[:, :4]),
    ('欠拟合（1阶）',     train_poly[:, :2], test_poly[:, :2]),
    ('过拟合（20阶）',    train_poly,        test_poly),
]

for ax, (title, tr_f, te_f) in zip(axes, configs):
    tr_ls, te_ls = train_poly(tr_f, te_f, train_labels, test_labels,
                              num_epochs=400, lr=0.01)
    iters = range(20, 401, 20)
    ax.semilogy(iters, tr_ls, label='训练')
    ax.semilogy(iters, te_ls, label='测试')
    ax.set_title(title)
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE（对数）')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('多项式回归：模型复杂度对泛化的影响', y=1.01)
plt.tight_layout(); plt.show()

**实验结论：**
- **3 阶（正常）**：训练与测试损失同步下降，泛化良好
- **1 阶（欠拟合）**：两者损失均高，模型无法捕捉规律
- **20 阶（过拟合）**：训练损失极低，测试损失居高不下，模型记住了噪声

---
## 4.4 权重衰减（Weight Decay）

### 4.4.1 原理

权重衰减是最常用的正则化技术，通过在损失函数中加入 **L2 范数惩罚项**限制参数取值范围：

$$\tilde{L}(\mathbf{w}, b) = L(\mathbf{w}, b) + \frac{\lambda}{2}\|\mathbf{w}\|^2$$

其中 $\lambda \geq 0$ 控制正则化强度。对应的梯度更新变为：

$$\mathbf{w} \leftarrow \underbrace{(1 - \eta\lambda)}_{\text{权重衰减因子}} \mathbf{w} - \frac{\eta}{|\mathcal{B}|}\sum_{i\in\mathcal{B}} \nabla_{\mathbf{w}} L^{(i)}$$

每次更新前权重乘以 $(1-\eta\lambda) < 1$，因此称为**权重衰减**。

### 4.4.2 实验：高维线性回归

In [ ]:
# 生成高维小样本数据（容易过拟合）
n_train, n_test, num_inputs = 20, 100, 200
true_w2 = torch.ones(num_inputs, 1) * 0.01
true_b2 = 0.05

def synthetic_data_wd(w, b, n):
    X = torch.normal(0, 1, (n, len(w)))
    y = X @ w + b + torch.normal(0, 0.01, (n, 1))
    return X, y.squeeze()

train_data = synthetic_data_wd(true_w2, true_b2, n_train)
test_data  = synthetic_data_wd(true_w2, true_b2, n_test)

def make_iter(feat, lab, bs=5):
    return data.DataLoader(data.TensorDataset(feat, lab), batch_size=bs, shuffle=True)

train_iter_wd = make_iter(*train_data)
test_iter_wd  = make_iter(*test_data, bs=100)

In [ ]:
def train_wd(wd_lambda, num_epochs=100, lr=0.003):
    """训练并返回训练/测试损失历史，通过优化器 weight_decay 实现 L2 正则"""
    net = nn.Linear(num_inputs, 1)
    nn.init.normal_(net.weight, std=1)
    nn.init.zeros_(net.bias)

    loss_fn = nn.MSELoss()
    # PyTorch 在优化器中直接支持 weight_decay
    optimizer = torch.optim.SGD([
        {'params': net.weight, 'weight_decay': wd_lambda},
        {'params': net.bias}   # 偏置通常不做正则化
    ], lr=lr)

    tr_ls, te_ls = [], []
    for epoch in range(num_epochs):
        net.train()
        for X, y in train_iter_wd:
            optimizer.zero_grad()
            loss_fn(net(X).squeeze(), y).backward()
            optimizer.step()

        net.eval()
        with torch.no_grad():
            tr_l = loss_fn(net(train_data[0]).squeeze(), train_data[1]).item()
            te_l = loss_fn(net(test_data[0]).squeeze(),  test_data[1]).item()
        tr_ls.append(tr_l); te_ls.append(te_l)

    print(f'λ={wd_lambda}  |  训练损失 {tr_ls[-1]:.4f}  测试损失 {te_ls[-1]:.4f}  '
          f'‖w‖ = {net.weight.norm().item():.4f}')
    return tr_ls, te_ls

print('--- 无正则化 ---')
tr_no, te_no = train_wd(0)
print('\n--- 有正则化 (λ=3) ---')
tr_wd, te_wd = train_wd(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, (tr, te), title in zip(
    axes,
    [(tr_no, te_no), (tr_wd, te_wd)],
    ['无权重衰减（λ=0）', '有权重衰减（λ=3）']):

    ax.semilogy(tr, label='训练')
    ax.semilogy(te, label='测试')
    ax.set_title(title)
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE（对数）')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('权重衰减对过拟合的抑制效果', y=1.01)
plt.tight_layout(); plt.show()

---
## 4.5 Dropout

### 4.5.1 原理

Dropout 在训练中以概率 $p$ **随机将隐藏单元置零**，并对保留的单元做**缩放补偿**，保证期望不变：

$$h' = \begin{cases} 0 & \text{以概率 } p \\ \dfrac{h}{1-p} & \text{以概率 } 1-p \end{cases}$$

因此 $\mathbb{E}[h'] = h$。**测试时关闭 dropout**，使用完整网络做预测。

**直觉**：迫使网络不依赖任意一组神经元，学习冗余、鲁棒的特征表示。等价于训练了大量共享参数的子网络的集成。

In [ ]:
# 可视化 dropout 的效果
torch.manual_seed(42)
h = torch.arange(1, 9, dtype=torch.float32)  # 8 个神经元的激活值

def dropout_layer(X, p):
    assert 0 <= p <= 1
    if p == 1: return torch.zeros_like(X)
    if p == 0: return X
    mask = (torch.rand(X.shape) > p).float()
    return mask * X / (1.0 - p)

print('原始激活:', h.numpy())
print('dropout=0.0:', dropout_layer(h, 0.0).numpy())
print('dropout=0.5:', dropout_layer(h, 0.5).numpy())
print('dropout=1.0:', dropout_layer(h, 1.0).numpy())

### 4.5.2 简洁实现

In [ ]:
# 使用 nn.Dropout 实现（靠近输入层用较小 p，深层用较大 p）
dropout1, dropout2 = 0.2, 0.5

net_drop = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256), nn.ReLU(),
    nn.Dropout(dropout1),          # 第一隐藏层后
    nn.Linear(256, 256), nn.ReLU(),
    nn.Dropout(dropout2),          # 第二隐藏层后
    nn.Linear(256, 10)
)

net_drop.apply(init_weights)
print(net_drop)
print(f'\n可训练参数量: {sum(p.numel() for p in net_drop.parameters()):,}')

In [ ]:
history_drop = train(net_drop, train_iter, test_iter, num_epochs=10, lr=0.5)

# 与无 dropout 的 MLP 对比
fig, ax = plt.subplots(figsize=(7, 4))
epochs = range(1, 11)
ax.plot(epochs, history_mlp['test_acc'],  marker='o', label='MLP（无 Dropout）')
ax.plot(epochs, history_drop['test_acc'], marker='s', label='MLP（有 Dropout）')
ax.set_xlabel('Epoch'); ax.set_ylabel('测试准确率')
ax.set_title('Dropout 对测试准确率的影响')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 4.6 前向传播与反向传播

### 4.6.1 前向传播

以含 L2 正则的单隐藏层 MLP 为例，设输入 $\mathbf{x}$，正则系数 $\lambda$：

| 步骤 | 公式 |
|------|------|
| 隐藏层线性 | $\mathbf{z} = \mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$ |
| 激活 | $\mathbf{h} = \phi(\mathbf{z})$ |
| 输出层线性 | $\mathbf{o} = \mathbf{W}^{(2)}\mathbf{h} + \mathbf{b}^{(2)}$ |
| 损失 | $l = \text{CrossEntropy}(\mathbf{o}, y)$ |
| 正则项 | $s = \frac{\lambda}{2}(\|\mathbf{W}^{(1)}\|_F^2 + \|\mathbf{W}^{(2)}\|_F^2)$ |
| 目标 | $J = l + s$ |

### 4.6.2 反向传播

链式法则**从输出层往回**计算每个参数的梯度：

$$\frac{\partial J}{\partial \mathbf{o}} = \frac{\partial l}{\partial \mathbf{o}}$$

$$\frac{\partial J}{\partial \mathbf{W}^{(2)}} = \frac{\partial J}{\partial \mathbf{o}} \mathbf{h}^\top + \lambda \mathbf{W}^{(2)}$$

$$\frac{\partial J}{\partial \mathbf{h}} = \left(\mathbf{W}^{(2)}\right)^\top \frac{\partial J}{\partial \mathbf{o}}$$

$$\frac{\partial J}{\partial \mathbf{z}} = \frac{\partial J}{\partial \mathbf{h}} \odot \phi'(\mathbf{z})$$

$$\frac{\partial J}{\partial \mathbf{W}^{(1)}} = \frac{\partial J}{\partial \mathbf{z}} \mathbf{x}^\top + \lambda \mathbf{W}^{(1)}$$

**训练内存远大于推理内存**：中间变量（$\mathbf{z}$、$\mathbf{h}$ 等）在前向传播时必须保留，用于反向传播计算梯度。

In [ ]:
# 演示：用 PyTorch autograd 验证手动梯度
torch.manual_seed(0)
W1 = torch.randn(4, 3, requires_grad=True)  # 隐藏层权重
W2 = torch.randn(2, 4, requires_grad=True)  # 输出层权重
x0 = torch.randn(3)
y0 = torch.tensor(1)  # 目标类别

# 前向传播
z = W1 @ x0
h = torch.relu(z)
o = W2 @ h
J = nn.CrossEntropyLoss()(o.unsqueeze(0), y0.unsqueeze(0))

# 自动反向传播
J.backward()

print(f'损失 J = {J.item():.4f}')
print(f'∂J/∂W1 形状: {W1.grad.shape}')
print(f'∂J/∂W2 形状: {W2.grad.shape}')
print(f'∂J/∂W2 值:\n{W2.grad}')

---
## 4.7 数值稳定性与模型初始化

### 4.7.1 梯度消失与梯度爆炸

在 $L$ 层网络中，反向传播的梯度包含 $L$ 个矩阵的乘积：

$$\frac{\partial \mathbf{o}}{\partial \mathbf{W}^{(l)}} = \mathbf{M}^{(L)} \cdots \mathbf{M}^{(l+1)} \mathbf{v}^{(l)}$$

- **梯度消失**：若每层 $\|\mathbf{M}^{(i)}\| < 1$（如 sigmoid 的导数最大仅 0.25），乘积指数级趋零，早期层参数几乎不更新。
- **梯度爆炸**：若每层 $\|\mathbf{M}^{(i)}\| > 1$，乘积指数级膨胀，优化不稳定。

In [ ]:
# 可视化 sigmoid 导数的消失现象
x_v = torch.linspace(-10, 10, 400)
sig = torch.sigmoid(x_v)
sig_grad = sig * (1 - sig)  # sigmoid'(x)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x_v.numpy(), sig.numpy(), color='steelblue')
axes[0].set_title('Sigmoid 函数')
axes[0].axhline(0.5, color='gray', ls='--', lw=0.8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_v.numpy(), sig_grad.numpy(), color='tomato')
axes[1].set_title("Sigmoid 导数（最大值 = 0.25）")
axes[1].axhline(0.25, color='gray', ls='--', lw=0.8, label='max=0.25')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Sigmoid 的梯度消失问题', y=1.01)
plt.tight_layout(); plt.show()

# 10层网络梯度消失模拟
grad = 0.25 ** 10
print(f'\n若每层 sigmoid 导数均为最大值 0.25，10 层后梯度约为: {grad:.2e}')

### 4.7.2 Xavier 初始化

**核心思想**：保持每层输出方差与输入方差相同（前向和反向传播均满足）。

对于全连接层（输入维度 $n_{\text{in}}$，输出维度 $n_{\text{out}}$），Xavier 初始化权重满足：

$$\sigma = \sqrt{\frac{2}{n_{\text{in}} + n_{\text{out}}}}$$

**两种实现：**
- 高斯：$\mathbf{W} \sim \mathcal{N}\left(0,\ \frac{2}{n_{\text{in}}+n_{\text{out}}}\right)$
- 均匀：$\mathbf{W} \sim \mathcal{U}\left(-\sqrt{\frac{6}{n_{\text{in}}+n_{\text{out}}}},\ \sqrt{\frac{6}{n_{\text{in}}+n_{\text{out}}}}\right)$

In [ ]:
# 对比不同初始化方式下各层激活值的分布
def get_activations(net_init_fn, n_layers=5):
    """前向传播一个随机输入，收集每层激活值分布"""
    layers = []
    x_ = torch.randn(1000, 256)  # 1000 个样本，256 维输入
    for _ in range(n_layers):
        W = net_init_fn(256, 256)
        x_ = torch.tanh(x_ @ W)  # 用 tanh 激活
        layers.append(x_.detach().numpy())
    return layers

fig, axes = plt.subplots(2, 5, figsize=(14, 5), sharey=False)

# 标准正态初始化（容易梯度消失）
layers_normal = get_activations(lambda ni, no: torch.randn(ni, no) * 0.01)
# Xavier 均匀初始化
def xavier_uniform(ni, no):
    bound = np.sqrt(6.0 / (ni + no))
    return torch.empty(ni, no).uniform_(-bound, bound)
layers_xavier = get_activations(xavier_uniform)

for i, (ax, layer) in enumerate(zip(axes[0], layers_normal)):
    ax.hist(layer.flatten(), bins=40, density=True, color='steelblue', alpha=0.8)
    ax.set_title(f'标准正态 Layer {i+1}', fontsize=9)
    ax.set_xlim(-1.1, 1.1)

for i, (ax, layer) in enumerate(zip(axes[1], layers_xavier)):
    ax.hist(layer.flatten(), bins=40, density=True, color='tomato', alpha=0.8)
    ax.set_title(f'Xavier Layer {i+1}', fontsize=9)
    ax.set_xlim(-1.1, 1.1)

plt.suptitle('激活值分布：标准正态初始化（上）vs Xavier 初始化（下）', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# PyTorch 内置初始化方法示例
net_init = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256), nn.ReLU(),
    nn.Linear(256, 256), nn.ReLU(),
    nn.Linear(256, 10)
)

def init_demo(m):
    if isinstance(m, nn.Linear):
        # Xavier 均匀初始化（适合 tanh/sigmoid）
        nn.init.xavier_uniform_(m.weight)
        # Kaiming 正态初始化（适合 ReLU）
        # nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
        nn.init.zeros_(m.bias)
        print(f'  {m}  权重方差: {m.weight.data.var():.6f}')

print('Xavier 均匀初始化后各层权重方差：')
net_init.apply(init_demo)

---
## 4.8 环境与分布偏移

### 4.8.1 三种主要类型

训练环境和部署环境的数据分布往往不完全一致，这会导致模型在实际应用中性能下降：

| 类型 | 变化的量 | 不变的量 | 典型例子 |
|------|---------|---------|----------|
| **协变量偏移** | $P(\mathbf{x})$ | $P(y\mid\mathbf{x})$ | 用卡通猫训练，在真实照片上测试 |
| **标签偏移** | $P(y)$ | $P(\mathbf{x}\mid y)$ | 疾病流行率随季节变化 |
| **概念偏移** | $P(y\mid\mathbf{x})$ | — | 词义随时代改变；各地方言 |

### 4.8.2 协变量偏移纠正

通过对训练样本重新加权来修正分布差异：

$$\mathbf{w}^* = \arg\min_\mathbf{w} \sum_{i=1}^n \beta_i l(f(\mathbf{x}_i), y_i), \quad \beta_i = \frac{p(\mathbf{x}_i)}{q(\mathbf{x}_i)}$$

重要性权重 $\beta_i$ 可用对数几率回归估计：将来自 $p$ 的样本标记为 1，来自 $q$ 的样本标记为 0，训练二元分类器后推导权重。

In [ ]:
# 模拟协变量偏移场景
np.random.seed(42)

# 训练分布：N(0, 1)
x_train_env = np.random.normal(0, 1, 1000)
# 测试（部署）分布：N(2, 0.5) —— 分布偏移！
x_test_env  = np.random.normal(2, 0.5, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：分布对比
axes[0].hist(x_train_env, bins=40, density=True, alpha=0.6, label='训练分布 $q(x)$')
axes[0].hist(x_test_env,  bins=20, density=True, alpha=0.6, label='测试分布 $p(x)$')
axes[0].set_title('协变量偏移示例')
axes[0].set_xlabel('x'); axes[0].set_ylabel('密度')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# 右图：重要性权重
from scipy.stats import norm
x_range = np.linspace(-3, 4, 300)
p = norm.pdf(x_range, 2, 0.5)   # 测试（目标）分布
q = norm.pdf(x_range, 0, 1.0)   # 训练分布
beta = p / (q + 1e-8)           # 重要性权重

axes[1].plot(x_range, beta, color='purple')
axes[1].set_title('重要性权重 $\\beta(x) = p(x)/q(x)$')
axes[1].set_xlabel('x'); axes[1].set_ylabel('$\\beta$')
axes[1].axhline(1, color='gray', ls='--', lw=0.8, label='$\\beta=1$（无偏移）')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('协变量偏移：训练样本需要按 β(x) 加权', y=1.01)
plt.tight_layout(); plt.show()

### 4.8.3 实践建议

| 场景 | 推荐策略 |
|------|----------|
| 协变量偏移 | 估计重要性权重 $\beta_i = p(\mathbf{x}_i)/q(\mathbf{x}_i)$ 后加权训练 |
| 标签偏移 | 用混淆矩阵反演估计目标标签分布，计算权重 $\beta_i = p(y_i)/q(y_i)$ |
| 概念偏移 | 持续收集新数据，滚动更新（微调）模型 |
| 通用 | 建立上线监控机制，检测预测分布变化作为报警信号 |

---
## 小结

```
线性模型（Softmax）
    ↓  加隐藏层 + 激活函数
多层感知机（MLP）
    ↓  可能过拟合
正则化：权重衰减 + Dropout
    ↓  梯度消失/爆炸
合理初始化（Xavier/Kaiming）+ ReLU 激活
    ↓  训练-部署分布不一致
分布偏移检测与纠正
```

| 技术 | 解决的问题 | PyTorch API |
|------|-----------|-------------|
| 隐藏层 + ReLU | 非线性表达能力 | `nn.Linear` + `nn.ReLU` |
| 权重衰减 | 过拟合 | `optimizer weight_decay` |
| Dropout | 过拟合 | `nn.Dropout(p)` |
| Xavier 初始化 | 梯度消失/爆炸 | `nn.init.xavier_uniform_` |
| Kaiming 初始化 | ReLU 网络的梯度问题 | `nn.init.kaiming_normal_` |